# Receipt Classification & Signature Detection
## Complete Interactive Pipeline

This notebook demonstrates the entire receipt classification system step-by-step.

## 1️⃣ Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install -q numpy pandas opencv-python tensorflow scikit-learn matplotlib seaborn requests

In [ ]:
# Import all libraries
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import smart_resize

print("✓ All libraries imported successfully!")

## 2️⃣ Load Configuration

In [ ]:
# Configuration
BASE_PATH = os.path.abspath('.')
DATA_PATH = os.path.join(BASE_PATH, 'data')
COMPLETE_PATH = os.path.join(DATA_PATH, 'complete')
INCOMPLETE_PATH = os.path.join(DATA_PATH, 'incomplete')
SIGNATURES_PATH = os.path.join(DATA_PATH, 'signatures')
NON_SIGNATURES_PATH = os.path.join(DATA_PATH, 'non_signatures')
MODELS_PATH = os.path.join(BASE_PATH, 'models')
RESULTS_PATH = os.path.join(BASE_PATH, 'results')

# Create directories
for path in [COMPLETE_PATH, INCOMPLETE_PATH, SIGNATURES_PATH, NON_SIGNATURES_PATH, MODELS_PATH, RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

# Parameters
TEST_SPLIT = 0.2
RANDOM_SEED = 42
CNN_EPOCHS = 30
TRANSFER_LEARNING_EPOCHS = 20
BATCH_SIZE = 16

print(f"✓ Configuration loaded")
print(f"  Data path: {DATA_PATH}")
print(f"  Results path: {RESULTS_PATH}")

## 3️⃣ Feature Extraction & Receipt Classifier

In [ ]:
# Keywords for field detection
MONTHS = "jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|september|oct|october|nov|november|dec|december"

def extract_features(text):
    """Extract 7 features from OCR text"""
    t = text.lower()
    
    has_date = int(bool(re.search(rf"({MONTHS})\s+\d{{1,2}}", t)))
    has_name = int(any(k in t for k in ['company', 'ltd', 'shop', 'restaurant', 'name', 'ชื่อ']))
    has_item = int(any(k in t for k in ['description', 'qty', 'amount', 'total', 'price', 'บาท']))
    has_sig = int(any(k in t for k in ['signature', 'received', 'ลายเซ็น', 'sign']))
    has_addr = int(any(k in t for k in ['address', 'mae fah luang', 'mfu']))
    text_len = len(t) / 100
    digit_cnt = sum(1 for c in t if c.isdigit()) / 10
    
    return [has_date, has_name, has_item, has_sig, has_addr, text_len, digit_cnt]

print("✓ Feature extraction function ready")

In [ ]:
# Receipt Classifier Class
class ReceiptClassifier:
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
        self.scaler = StandardScaler()
    
    def train(self, X_train, y_train):
        X_scaled = self.scaler.fit_transform(X_train)
        self.model.fit(X_scaled, y_train)
        print("✓ Receipt classifier trained")
    
    def evaluate(self, X_test, y_test):
        X_scaled = self.scaler.transform(X_test)
        y_pred = self.model.predict(X_scaled)
        
        return {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred),
            'confusion_matrix': confusion_matrix(y_test, y_pred),
            'y_pred': y_pred
        }

print("✓ ReceiptClassifier class ready")

## 4️⃣ Create Dataset

In [ ]:
# Create synthetic dataset for demo
# (In production, this would come from actual receipt images processed with Typhoon OCR)

sample_texts_complete = [
    "Company Name Ltd, Date: March 15, 2024, Item: Coffee $5, Tea $3, Signature: John, Address: Mae Fah Luang",
    "Restaurant ABC, January 20, 2024, Qty: 2, Amount: $25, Price: $12.50, Received: True, Address: MFU",
]

sample_texts_incomplete = [
    "Total: $50",
    "Some text without complete information",
]

# Extract features
features_list = []
labels_list = []

# Complete receipts
for text in sample_texts_complete:
    features = extract_features(text)
    features_list.append(features)
    labels_list.append(1)

# Incomplete receipts
for text in sample_texts_incomplete:
    features = extract_features(text)
    features_list.append(features)
    labels_list.append(0)

# Add more samples for better training
for i in range(4):
    # Complete-like features
    features_list.append([1, 1, 1, 1, 1, 5+i, 15+i*2])
    labels_list.append(1)
    # Incomplete-like features
    features_list.append([i%2, i%2, 1, 0, 0, 2+i, 5+i])
    labels_list.append(0)

X = np.array(features_list)
y = np.array(labels_list)

print(f"✓ Dataset created: {len(X)} samples")
print(f"  Features shape: {X.shape}")
print(f"  Classes: {np.unique(y)}")
print(f"  Class distribution: Complete={np.sum(y)}, Incomplete={len(y)-np.sum(y)}")

## 5️⃣ Train Receipt Classifier

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED, stratify=y
)

print(f"✓ Data split:")
print(f"  Training: {len(X_train)} samples")
print(f"  Testing: {len(X_test)} samples")

# Train
classifier = ReceiptClassifier()
classifier.train(X_train, y_train)

# Evaluate
receipt_metrics = classifier.evaluate(X_test, y_test)

print(f"\n📊 Receipt Classifier Results:")
print(f"  Accuracy:  {receipt_metrics['accuracy']:.4f}")
print(f"  Precision: {receipt_metrics['precision']:.4f}")
print(f"  Recall:    {receipt_metrics['recall']:.4f}")
print(f"  F1-Score:  {receipt_metrics['f1']:.4f}")

## 6️⃣ Build Signature Detection Models

In [ ]:
# CNN Model
def build_cnn_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("✓ CNN model defined")

In [ ]:
# Transfer Learning Model
def build_transfer_learning_model():
    base_model = ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("✓ Transfer Learning model defined")

## 7️⃣ Create Signature Dataset

In [ ]:
# Create synthetic signature data (in production, load from data/signatures and data/non_signatures)
X_sig = np.random.rand(10, 128, 128, 3)  # 10 signature samples
X_nonsig = np.random.rand(10, 128, 128, 3)  # 10 non-signature samples

# Combine and label
X_combined = np.vstack([X_sig, X_nonsig])
y_combined = np.array([1] * len(X_sig) + [0] * len(X_nonsig))

# Split
X_train_sig, X_test_sig, y_train_sig, y_test_sig = train_test_split(
    X_combined, y_combined, test_size=TEST_SPLIT, random_state=RANDOM_SEED, stratify=y_combined
)

print(f"✓ Signature dataset created:")
print(f"  Total samples: {len(X_combined)}")
print(f"  Training: {len(X_train_sig)}, Testing: {len(X_test_sig)}")
print(f"  Class distribution: Signatures={np.sum(y_combined)}, Non-signatures={len(y_combined)-np.sum(y_combined)}")

## 8️⃣ Train CNN Model

In [ ]:
print("🔄 Training CNN Model...")
cnn_model = build_cnn_model()
cnn_history = cnn_model.fit(
    X_train_sig, y_train_sig,
    validation_split=0.2,
    epochs=CNN_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0
)

cnn_pred = (cnn_model.predict(X_test_sig, verbose=0) > 0.5).astype(int).flatten()
cnn_metrics = {
    'accuracy': accuracy_score(y_test_sig, cnn_pred),
    'precision': precision_score(y_test_sig, cnn_pred, zero_division=0),
    'recall': recall_score(y_test_sig, cnn_pred, zero_division=0),
    'f1': f1_score(y_test_sig, cnn_pred, zero_division=0),
}

print(f"✓ CNN Training Complete")
print(f"  Accuracy:  {cnn_metrics['accuracy']:.4f}")
print(f"  Precision: {cnn_metrics['precision']:.4f}")
print(f"  Recall:    {cnn_metrics['recall']:.4f}")
print(f"  F1-Score:  {cnn_metrics['f1']:.4f}")

## 9️⃣ Train Transfer Learning Model

In [ ]:
print("🔄 Training Transfer Learning Model (ResNet50)...")

# Resize images to 224x224
X_train_tl = np.array([smart_resize(img, (224, 224)) for img in X_train_sig])
X_test_tl = np.array([smart_resize(img, (224, 224)) for img in X_test_sig])

tl_model = build_transfer_learning_model()
tl_history = tl_model.fit(
    X_train_tl, y_train_sig,
    validation_split=0.2,
    epochs=TRANSFER_LEARNING_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0
)

tl_pred = (tl_model.predict(X_test_tl, verbose=0) > 0.5).astype(int).flatten()
tl_metrics = {
    'accuracy': accuracy_score(y_test_sig, tl_pred),
    'precision': precision_score(y_test_sig, tl_pred, zero_division=0),
    'recall': recall_score(y_test_sig, tl_pred, zero_division=0),
    'f1': f1_score(y_test_sig, tl_pred, zero_division=0),
}

print(f"✓ Transfer Learning Training Complete")
print(f"  Accuracy:  {tl_metrics['accuracy']:.4f}")
print(f"  Precision: {tl_metrics['precision']:.4f}")
print(f"  Recall:    {tl_metrics['recall']:.4f}")
print(f"  F1-Score:  {tl_metrics['f1']:.4f}")

## 🔟 Results & Visualization

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Receipt Classification & Signature Detection Results', fontsize=16, fontweight='bold')

# Receipt Classifier Confusion Matrix
cm_receipt = receipt_metrics['confusion_matrix']
sns.heatmap(cm_receipt, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Incomplete', 'Complete'], yticklabels=['Incomplete', 'Complete'])
axes[0, 0].set_title('Receipt Classifier\nConfusion Matrix', fontweight='bold')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# Model Performance Comparison
models = ['Receipt\nClassifier', 'CNN', 'Transfer\nLearning']
accuracies = [receipt_metrics['accuracy'], cnn_metrics['accuracy'], tl_metrics['accuracy']]
f1_scores = [receipt_metrics['f1'], cnn_metrics['f1'], tl_metrics['f1']]
precisions = [receipt_metrics['precision'], cnn_metrics['precision'], tl_metrics['precision']]
recalls = [receipt_metrics['recall'], cnn_metrics['recall'], tl_metrics['recall']]

x = np.arange(len(models))
width = 0.2

axes[0, 1].bar(x - 1.5*width, accuracies, width, label='Accuracy', color='#2ecc71')
axes[0, 1].bar(x - 0.5*width, precisions, width, label='Precision', color='#3498db')
axes[0, 1].bar(x + 0.5*width, recalls, width, label='Recall', color='#e74c3c')
axes[0, 1].bar(x + 1.5*width, f1_scores, width, label='F1-Score', color='#f39c12')

axes[0, 1].set_ylabel('Score', fontweight='bold')
axes[0, 1].set_title('Model Metrics Comparison', fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models)
axes[0, 1].legend(loc='lower right')
axes[0, 1].set_ylim([0, 1.1])
axes[0, 1].grid(axis='y', alpha=0.3, linestyle='--')

# CNN Confusion Matrix
cm_cnn = confusion_matrix(y_test_sig, cnn_pred)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Reds', ax=axes[1, 0],
            xticklabels=['Non-Sig', 'Signature'], yticklabels=['Non-Sig', 'Signature'])
axes[1, 0].set_title('CNN Confusion Matrix', fontweight='bold')
axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')

# Transfer Learning Confusion Matrix
cm_tl = confusion_matrix(y_test_sig, tl_pred)
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', ax=axes[1, 1],
            xticklabels=['Non-Sig', 'Signature'], yticklabels=['Non-Sig', 'Signature'])
axes[1, 1].set_title('Transfer Learning (ResNet50)\nConfusion Matrix', fontweight='bold')
axes[1, 1].set_ylabel('True Label')
axes[1, 1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'results.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to results/results.png")

## 1️⃣1️⃣ Export Metrics to CSV

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Model': ['Receipt Classifier', 'Receipt Classifier', 'Receipt Classifier', 'Receipt Classifier',
              'CNN', 'CNN', 'CNN', 'CNN',
              'Transfer Learning', 'Transfer Learning', 'Transfer Learning', 'Transfer Learning'],
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'] * 3,
    'Value': [
        receipt_metrics['accuracy'], receipt_metrics['precision'], receipt_metrics['recall'], receipt_metrics['f1'],
        cnn_metrics['accuracy'], cnn_metrics['precision'], cnn_metrics['recall'], cnn_metrics['f1'],
        tl_metrics['accuracy'], tl_metrics['precision'], tl_metrics['recall'], tl_metrics['f1']
    ]
})

results_df.to_csv(os.path.join(RESULTS_PATH, 'metrics.csv'), index=False)

print("✓ Metrics exported to results/metrics.csv\n")
print("📊 Metrics Summary:")
print(results_df.to_string(index=False))

## 1️⃣2️⃣ Final Summary

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print(f"\n📋 Receipt Classification")
print(f"   Accuracy:  {receipt_metrics['accuracy']:.2%}")
print(f"   Precision: {receipt_metrics['precision']:.2%}")
print(f"   Recall:    {receipt_metrics['recall']:.2%}")
print(f"   F1-Score:  {receipt_metrics['f1']:.2%}")

print(f"\n🖼️  CNN Signature Detection")
print(f"   Accuracy:  {cnn_metrics['accuracy']:.2%}")
print(f"   Precision: {cnn_metrics['precision']:.2%}")
print(f"   Recall:    {cnn_metrics['recall']:.2%}")
print(f"   F1-Score:  {cnn_metrics['f1']:.2%}")

print(f"\n🔄 Transfer Learning (ResNet50)")
print(f"   Accuracy:  {tl_metrics['accuracy']:.2%}")
print(f"   Precision: {tl_metrics['precision']:.2%}")
print(f"   Recall:    {tl_metrics['recall']:.2%}")
print(f"   F1-Score:  {tl_metrics['f1']:.2%}")

print(f"\n✓ Results saved to: {RESULTS_PATH}")
print(f"  - results.png (visualizations)")
print(f"  - metrics.csv (detailed metrics)")

print("\n" + "="*70)
print("🎉 Pipeline completed successfully!")
print("="*70)